In [ ]:
import json
import os
import sys

module_path = os.path.abspath(os.path.join("..", "../ophir"))

# Add the path to sys.path
if module_path not in sys.path:
    sys.path.insert(0, module_path)
os.environ["TORCH_LOGS"] = "recompiles"

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from ophir.coin_datasets import construct_datasets
from ophir.training_models import LightningMulitClassPricePredictor, LightningPricePredictor

torch.set_float32_matmul_precision("high")
torch._functorch.config.donated_buffer = False
%matplotlib inline

In [ ]:
dir_path = "../data"
min_value = -10
max_value = 10
elements_per_sample = 180
elements_in_encoder = 150
train_loader, test_loader, sector_tokens, stock_tokens = construct_datasets(
    dir_path, "h", min_value, max_value, elements_per_sample, elements_in_encoder, quicktest=True
)

In [ ]:
ckpt_path = "saved-models/ophir-small-stock-weights-modifiers-v12.ckpt"
model = LightningMulitClassPricePredictor.load_from_checkpoint(ckpt_path)
trainer = L.Trainer(accelerator="cuda", precision="16-mixed")
model = model.cuda()
for inputs, _ in test_loader:
    model_output = model((input.cuda() for input in inputs))
    break


In [ ]:
output = trainer.predict(model, test_loader)
unzipped_data = zip(*output, strict=False)
model_output, target = [list(item) for item in unzipped_data]
model_output, target = (
    torch.concat(model_output, dim=0).squeeze(-1),
    torch.concat(target, dim=0).squeeze(-1),
)
print(model_output.shape, target.shape)
residuals = (model_output - target).abs().mean(0).cpu().numpy()
plt.figure(figsize=(12, 12))
plt.bar(list(range(len(residuals))), residuals)
plt.plot(
    list(range(len(residuals))),
    np.zeros_like(residuals),
    color="black",
)
plt.xlabel("Day")
plt.ylim(0, 1)
plt.ylabel("Avg. Absolute Residual")
plt.show()
plt.close()


In [ ]:
true_changes = target[0].cpu().numpy()  # first 180 in test set
output = model_output[0].cpu().numpy()  # first 180 in test set
plt.figure(figsize=(12, 12))
plt.plot(
    list(range(len(true_changes))),
    true_changes,
    color="g",
    alpha=0.5,
)
plt.plot(
    list(range(len(true_changes))),
    np.zeros_like(true_changes),
    color="black",
)
plt.plot(
    list(range(len(output))),
    output,
    color="b",
)
plt.xlabel("Day")
plt.ylim(-5, 5)
plt.ylabel("Percentange Change")


plt.show()
plt.close()